# Stage 1B: Cointegration (Engle-Granger) Pipeline
**Strategy**: NSE Intraday Pairs Trading  
**Purpose**: Run Augmented Dickey-Fuller (ADF) tests on the exact residuals (spreads) generated by:
1. Kalman Worst-Case
2. Kalman Dominant Regime
3. Rolling 20-Day OLS


In [ ]:
import os, glob, gc, json, shutil
import sqlite3
import pandas as pd
import numpy as np
from scipy.stats import t as t_dist
from statsmodels.tsa.stattools import adfuller

print("=== /kaggle/input contents ===")
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        fpath = os.path.join(root, f)
        print(f"  {fpath}  ({os.path.getsize(fpath)/(1024**3):.2f} GB)")

hits = glob.glob('/kaggle/input/**/*.sqlite', recursive=True)
if not hits:
    raise FileNotFoundError("No .sqlite found under /kaggle/input")
DB_PATH = hits[0]
print(f"\n✅ DB_PATH = {DB_PATH}")


In [ ]:
print("=== Loading All DB Data ===")
con = sqlite3.connect(DB_PATH)
df = pd.read_sql("SELECT symbol, timestamp, close FROM ohlcv_1min ORDER BY timestamp", con)
con.close()

df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert('Asia/Kolkata')
time_int = df['dt'].dt.hour * 100 + df['dt'].dt.minute
df_trading = df[(time_int >= 915) & (time_int <= 1529)].copy()
del df; gc.collect()

print("Pivoting to price matrix...")
price_matrix = df_trading.pivot(index='dt', columns='symbol', values='close')
del df_trading; gc.collect()

log_prices = np.log(price_matrix)
print(f"Price matrix: {price_matrix.shape}")


## Stage 1 — Pearson Correlation Screening
**Output**: `pairs_top500.csv`


In [ ]:
log_returns = log_prices - log_prices.shift(1)
dates_arr = np.array(price_matrix.index.date)
session_open_mask = np.concatenate([[True], dates_arr[1:] != dates_arr[:-1]])
log_returns.iloc[session_open_mask] = np.nan

print("Computing pairwise Pearson correlation...")
corr_df = log_returns.corr(method='pearson')

symbols = corr_df.columns.tolist()
rows = []
for i in range(len(symbols)):
    for j in range(i + 1, len(symbols)):
        rho = corr_df.iloc[i, j]
        if np.isnan(rho): continue
        n_pair = log_returns[[symbols[i], symbols[j]]].dropna().shape[0]
        if n_pair < 5000: continue
        t_stat = rho * np.sqrt((n_pair - 2) / max(1.0 - rho**2, 1e-12))
        p_val  = 2 * t_dist.sf(abs(t_stat), df=n_pair - 2)
        if p_val >= 0.05: continue
        rows.append({
            "symbol_a": symbols[i], "symbol_b": symbols[j],
            "pearson_rho": round(rho, 6)
        })

pairs_df = pd.DataFrame(rows).sort_values("pearson_rho", ascending=False).reset_index(drop=True)
pairs_df.head(500).to_csv("pairs_top500.csv", index=False)
print("Saved Stage 1 outputs.")

top500 = pairs_df.head(500)
TOP_PAIRS = list(zip(top500["symbol_a"], top500["symbol_b"]))
print(f"\nUsing Top {len(TOP_PAIRS)} Production Pairs for Stage 1B...")


## Stage 1B — Residual Generation & ADF Test
**Output**: `stage1b_cointegration_results.csv`


In [ ]:
NUM_CHUNKS = 4
WARMUP_BARS = 1875

def find_medoid(half_lives):
    hls = np.array(half_lives)
    if len(hls) == 1: return hls[0]
    distances = np.array([np.sum(np.abs(hl - hls)) for hl in hls])
    return hls[np.argmin(distances)]

def extract_ou_distribution(ya, yb, num_chunks):
    chunk_size = len(ya) // num_chunks
    valid_hls = []
    for i in range(num_chunks):
        start = i * chunk_size
        end = (i + 1) * chunk_size if i < num_chunks - 1 else len(ya)
        y_c, x_c = ya[start:end], yb[start:end]
        X_mat = np.column_stack([x_c, np.ones(len(x_c))])
        beta, _, _, _ = np.linalg.lstsq(X_mat, y_c, rcond=None)
        spread = y_c - X_mat @ beta
        X_ar = np.column_stack([spread[:-1], np.ones(len(spread) - 1)])
        phi_res, _, _, _ = np.linalg.lstsq(X_ar, spread[1:], rcond=None)
        phi = phi_res[0]
        if 0 < phi < 1:
            valid_hls.append(-np.log(2) / np.log(phi))
    if not valid_hls: return None
    return {
        "hl_max": float(np.max(valid_hls)),
        "hl_medoid": float(find_medoid(valid_hls))
    }

def compute_q_from_tau(target_tau, ya_warmup, yb_warmup):
    n = len(ya_warmup)
    X_w = np.column_stack([yb_warmup, np.ones(n)])
    beta0 = np.linalg.lstsq(X_w, ya_warmup, rcond=None)[0]
    res = ya_warmup - X_w @ beta0
    R_est = np.sum(res ** 2) / (n - 2)
    K_factor = 1.0 - np.power(0.5, 1.0 / target_tau)
    lam = (K_factor ** 2) / (1.0 - K_factor)
    Sigma_X_inv = np.linalg.inv(X_w.T @ X_w / n)
    Q = lam * R_est * Sigma_X_inv
    P0 = R_est * np.linalg.inv(X_w.T @ X_w)
    return Q, P0, R_est

def run_kalman_filter(ya, yb, timestamps, Q, P0, R):
    T = len(ya)
    X_full = np.column_stack([yb, np.ones(T)])
    x_upd = np.linalg.lstsq(X_full[:min(100, T)], ya[:min(100, T)], rcond=None)[0]
    P_upd = P0.copy()
    time_int = timestamps.hour * 100 + timestamps.minute
    is_open = (time_int == 915)
    spread = np.zeros(T)
    for t in range(T):
        x_p = x_upd
        P_p = P_upd + Q
        if is_open[t]: P_p *= 2.0
        H_t = X_full[t]
        v_t = ya[t] - H_t @ x_p
        S_t = H_t @ P_p @ H_t + R
        K_t = P_p @ H_t / S_t
        x_upd = x_p + K_t * v_t
        P_upd = P_p - np.outer(K_t, H_t) @ P_p
        spread[t] = v_t
    return spread

def get_adf(spread):
    try:
        clean_spread = spread[~np.isnan(spread)]
        if len(clean_spread) < 100: return np.nan, np.nan
        res = adfuller(clean_spread, maxlag=1)
        return res[0], res[1]  # adf_stat, p_value
    except:
        return np.nan, np.nan

results_st1b = []
daily_closes = price_matrix.groupby(price_matrix.index.date).last()
dates = daily_closes.index.values

for sym_a, sym_b in TOP_PAIRS:
    df_pair = log_prices[[sym_a, sym_b]].dropna(how='any')
    ya, yb, times = df_pair[sym_a].values, df_pair[sym_b].values, df_pair.index
    warmup_n = min(WARMUP_BARS, len(ya) // 10)
    
    ou = extract_ou_distribution(ya, yb, NUM_CHUNKS)
    if not ou: continue
    
    # Kalman Worst-Case Spread
    Q_wc, P0_wc, R_wc = compute_q_from_tau(ou["hl_max"] * 2.0, ya[:warmup_n], yb[:warmup_n])
    spread_wc = run_kalman_filter(ya, yb, times, Q_wc, P0_wc, R_wc)
    adf_wc, pval_wc = get_adf(spread_wc)
    
    # Kalman Dominant-Regime Spread
    Q_dr, P0_dr, R_dr = compute_q_from_tau(ou["hl_medoid"] * 2.0, ya[:warmup_n], yb[:warmup_n])
    spread_dr = run_kalman_filter(ya, yb, times, Q_dr, P0_dr, R_dr)
    adf_dr, pval_dr = get_adf(spread_dr)
    
    # OLS 20-Day Spread
    spread_ols = np.full(len(ya), np.nan)
    intraday_dates = times.date
    for d_idx in range(20, len(dates)):
        current_date = dates[d_idx]
        prev_20_dates = dates[d_idx-20:d_idx]
        y_daily = daily_closes.loc[prev_20_dates, sym_a].values
        x_daily = daily_closes.loc[prev_20_dates, sym_b].values
        if np.isnan(y_daily).any() or np.isnan(x_daily).any(): continue
        X_mat = np.column_stack([x_daily, np.ones(len(x_daily))])
        try:
            beta, alpha = np.linalg.lstsq(X_mat, y_daily, rcond=None)[0]
            mask = (intraday_dates == current_date)
            spread_ols[mask] = ya[mask] - (alpha + beta * yb[mask])
        except:
            continue
            
    adf_ols, pval_ols = get_adf(spread_ols)
    
    results_st1b.append({
        "pair": f"{sym_a}-{sym_b}",
        "wc_adf_stat": round(adf_wc, 4), "wc_pval": round(pval_wc, 6),
        "dr_adf_stat": round(adf_dr, 4), "dr_pval": round(pval_dr, 6),
        "ols_adf_stat": round(adf_ols, 4), "ols_pval": round(pval_ols, 6),
    })

res_df = pd.DataFrame(results_st1b)
res_df.to_csv("stage1b_cointegration_results.csv", index=False)
print("Saved Stage 1B outputs.")
display(res_df)


## Publish Output Dataset


In [ ]:
import json, os, shutil
from kaggle.api.kaggle_api_extended import KaggleApi

os.environ['KAGGLE_USERNAME'] = 'utkarshpatelthefirst'
os.environ['KAGGLE_KEY'] = 'fbef16329099428205f671dd5de8337b'

api = KaggleApi()
api.authenticate()

export_dir = '/kaggle/working/dataset_export'
os.makedirs(export_dir, exist_ok=True)

if os.path.exists('stage1b_cointegration_results.csv'):
    shutil.copy('stage1b_cointegration_results.csv', f'{export_dir}/stage1b_cointegration_results.csv')

meta = {
    "title"    : "Pairs Stage1B Cointegration v1",
    "id"       : "utkarshpatelthefirst/pairs-stage1b-cointegration-v1",
    "licenses" : [{"name": "CC0-1.0"}]
}
with open(f'{export_dir}/dataset-metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

try:
    api.dataset_create_new(export_dir, dir_mode='zip', quiet=False)
except Exception as e:
    if "already exists" in str(e).lower() or "409" in str(e):
        print("Dataset exists, updating version...")
        api.dataset_create_version(export_dir, "Update", dir_mode='zip', quiet=False)
    else:
        raise e

print("✅ Dataset ready and published.")
